In [ ]:

import datetime
import numpy as np
import cv2
from itertools import cycle
import pickle
import tqdm
from matplotlib import pyplot as plt
import scipy.signal as sig
import pandas as pd
from scipy.stats import kde
from eye_tracking_system_tools.preprocessing.BlockSync_class import BlockSync
from eye_tracking_system_tools.preprocessing import utility_functions as uf
from eye_tracking_system_tools.preprocessing.notebook_helpers import (
    create_distance_plot,
    add_intermediate_elements,
    find_jittery_frames,
    bokeh_plotter,
    rolling_window_analysis,
    create_behavior_df,
)
from scipy import signal
import bokeh
import seaborn as sns
from matplotlib import rcParams
from pathlib import Path
%matplotlib inline
plt.style.use('default')
rcParams['pdf.fonttype'] = 42  # Ensure fonts are embedded and editable
rcParams['ps.fonttype'] = 42  # Ensure compatibility with vector outputs

In [ ]:
# The helpers create_distance_plot, add_intermediate_elements,
# find_jittery_frames and bokeh_plotter that used to live here are now
# in eye_tracking_system_tools.preprocessing.notebook_helpers and are
# imported in Cell 0. get_frame_count was unused downstream and has
# been removed.


In [ ]:

# define a single block to synchronize and finally export l/r_eye_data csv files:
# this step creates block_collection - a list of BlockSync objects of interest
block_numbers = [15]
bad_blocks = [] #
experiment_path = Path(r"D:\sample_data_for_eye_repo")
animal = 'PV_106'

block_collection = uf.block_generator(block_numbers=block_numbers,
                                      experiment_path=experiment_path,
                                      animal=animal,
                                      bad_blocks=bad_blocks)
for block in block_collection:
    block.channeldict = None
    if block.animal_call == 'PV_208':
        block.channeldict={1: 'LED_driver',
                           7: 'L_eye_TTL',
                           2: 'Arena_TTL',
                           8: 'R_eye_TTL'}
# create a block_dict object for ease of access:
block_dict = {}
for b in block_collection:
    block_dict[str(b.block_num)] = b
block = block_collection[0]

In [ ]:
block = block_collection[0]

In [ ]:
block.block_get_lizard_movement()

In [ ]:
# Rolling-window analysis + interactive threshold selection.
# `rolling_window_analysis` now lives in notebook_helpers (imported in Cell 0).
import plotly.graph_objects as go

df = rolling_window_analysis(block.liz_mov_df, 10000, 1000)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df['window_start'],
    y=df['average_movAll'],
    mode='lines+markers',
    name='Average Movement',
    line=dict(color='blue')
))

initial_threshold = 0.5
fig.add_trace(go.Scatter(
    x=df['window_start'],
    y=[initial_threshold] * len(df),
    mode='lines',
    name='Threshold',
    line=dict(color='red', dash='dash')
))

fig.update_layout(
    title='Interactive Movement Threshold Selection',
    xaxis_title='Time (ms)',
    yaxis_title='Average Movement',
    sliders=[{
        'active': 5,
        'currentvalue': {'prefix': 'Threshold: '},
        'steps': [
            {'label': str(round(threshold, 2)), 'method': 'update',
             'args': [{'y': [df['average_movAll'], [threshold] * len(df)]}]}
            for threshold in [x / 10.0 for x in range(0, 20)]
        ]
    }]
)

fig.show()


In [ ]:
# Threshold and segmentation. `create_behavior_df` is imported from
# notebook_helpers in Cell 0.
threshold = 0.3
df['behavior'] = df['average_movAll'].apply(lambda x: 'active' if x > threshold else 'quiet')

behavior_df = create_behavior_df(df)
print(behavior_df)


In [ ]:
block.behavior_state = behavior_df
csv_path = block.analysis_path / f"block_{block.block_num}_behavior_state.csv"
behavior_df.to_csv(csv_path, index=False)
print(f"Behavior state saved to {csv_path}")